In [2]:
import sys
from pathlib import Path

parent_dir = str(Path().resolve().parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

In [3]:
from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"

from pprint import pformat

formatter = get_ipython().display_formatter.formatters['text/plain']
formatter.for_type(dict, lambda obj, p, _: p.text(pformat(obj, width=100, sort_dicts=False)))
formatter.for_type(list, lambda obj, p, _: p.text(pformat(obj, width=100, sort_dicts=False)))

<function IPython.lib.pretty._seq_pprinter_factory.<locals>.inner(obj, p, cycle)>

In [4]:
from datasets import load_from_disk

cfact = load_from_disk("../data/cfact")
cfact
cfact[0]

Dataset({
    features: ['subject', 'prompt', 'old_ans', 'new_ans', 'rephrases', 'locality'],
    num_rows: 20775
})

{'subject': 'Danielle Darrieux',
 'prompt': 'The mother tongue of Danielle Darrieux is',
 'old_ans': 'French',
 'new_ans': 'English',
 'rephrases': ["Danielle Darrieux's mother tongue is",
               'Danielle Darrieux was born in',
               'Where Danielle Darrieux is from, people speak the language of'],
 'locality': [{'ans': 'French', 'prompt': 'François Bayrou, a native'},
              {'ans': 'French', 'prompt': 'Maurice Genevoix, speaker of'},
              {'ans': 'French', 'prompt': 'Jacques Chaban-Delmas is a native speaker of'},
              {'ans': 'French', 'prompt': 'Michel Rocard is a native speaker of'},
              {'ans': 'French', 'prompt': 'The native language of Montesquieu is'},
              {'ans': 'French', 'prompt': 'The mother tongue of François Bayrou is'},
              {'ans': 'French', 'prompt': 'The native language of Raymond Barre is'},
              {'ans': 'French', 'prompt': 'The native language of François Bayrou is'},
              {'a

In [ ]:
import random
from dataclasses import asdict
from src.utils import EditRequest, EvalPair, EvalSuite
from evaluate.evaluate_temp import save_results

n = 20  # number of samples — change as needed

cfact_indices = random.sample(range(len(cfact)), n)
print(f"cfact indices: {cfact_indices}")

cfact indices: [12456, 11459, 10694, 8066, 17977, 17214, 2246, 2397, 9429, 2887, 9151, 13989, 20070, 11450, 5027, 9983, 16766, 1727, 1625, 18895]


In [6]:
from src.editors import load_editor
editor = load_editor("../src/config/ft/gpt2-xl.yaml")

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

In [7]:
cfact_results = []

for i, idx in enumerate(cfact_indices):
    sample = cfact[idx]
    request = EditRequest(
        subject=sample["subject"],
        prompt=sample["prompt"],
        old_ans=sample["old_ans"],
        new_ans=sample["new_ans"],
    )
    editor.edit(request, is_sequential=True)

    # Reliability: original prompt + rephrases, all expecting new_ans
    reliability_pairs = [EvalPair(prompt=sample["prompt"], ans=sample["new_ans"])]
    for rephrase in sample["rephrases"]:
        reliability_pairs.append(EvalPair(prompt=rephrase, ans=sample["new_ans"]))

    # Locality: unrelated facts that should stay unchanged
    locality_pairs = [EvalPair(prompt=loc["prompt"], ans=loc["ans"])
                      for loc in sample["locality"]]

    suite = EvalSuite(
        reliability=reliability_pairs,
        locality={"locality": locality_pairs} if locality_pairs else None,
    )
    result = editor.eval(suite, max_new_tokens=10)

    cfact_results.append({
        "edit_index": i + 1,
        "dataset_index": idx,
        "edit_request": asdict(request),
        **result.to_dict(),
    })
    rel_score = result.reliability.score if result.reliability else "N/A"
    loc_score = (list(result.locality.values())[0].score
                 if result.locality else "N/A")
    print(f"[cfact {i+1}/{n}] reliability={rel_score}, locality={loc_score}")

print("cfact phase done.")

Executing FT algo for: [Bani River is located in] -> [ Antarctica]
Weights to be updated: ['transformer.h.0.mlp.c_proj.weight', 'transformer.h.0.mlp.c_proj.bias']
==================== Epoch: 0 ====================
Batch loss 12.060379028320312
Total loss 12.060379028320312
==================== Epoch: 1 ====================
Batch loss 8.463622093200684
Total loss 8.463622093200684
==================== Epoch: 2 ====================
Batch loss 8.05457592010498
Total loss 8.05457592010498
==================== Epoch: 3 ====================
Batch loss 7.23958158493042
Total loss 7.23958158493042
==================== Epoch: 4 ====================
Batch loss 6.3080949783325195
Total loss 6.3080949783325195
==================== Epoch: 5 ====================
Batch loss 5.7036333084106445
Total loss 5.7036333084106445
==================== Epoch: 6 ====================
Batch loss 4.8978376388549805
Total loss 4.8978376388549805
==================== Epoch: 7 ====================
Batch loss 4.259852

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


Total loss 0.013422482647001743
Deltas successfully computed for ['transformer.h.0.mlp.c_proj.weight', 'transformer.h.0.mlp.c_proj.bias']
New weights successfully inserted into ['transformer.h.0.mlp.c_proj.weight', 'transformer.h.0.mlp.c_proj.bias']


A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


[cfact 1/20] reliability=0.5, locality=0.5
Executing FT algo for: [Nanticoke, Ontario, in] -> [ Ukraine]
Weights to be updated: ['transformer.h.0.mlp.c_proj.weight', 'transformer.h.0.mlp.c_proj.bias']
==================== Epoch: 0 ====================
Batch loss 11.201127052307129
Total loss 11.201127052307129
==================== Epoch: 1 ====================
Batch loss 9.192377090454102
Total loss 9.192377090454102
==================== Epoch: 2 ====================
Batch loss 8.476943016052246
Total loss 8.476943016052246
==================== Epoch: 3 ====================
Batch loss 5.39689826965332
Total loss 5.39689826965332
==================== Epoch: 4 ====================
Batch loss 2.9353554248809814
Total loss 2.9353554248809814
==================== Epoch: 5 ====================
Batch loss 1.9290058612823486
Total loss 1.9290058612823486
==================== Epoch: 6 ====================
Batch loss 1.4773820638656616
Total loss 1.4773820638656616
==================== Epoch: 7 

...

[cfact 19/20] reliability=0.75, locality=0.25
Executing FT algo for: [Shaad Ali, a citizen of] -> [ Albania]
Weights to be updated: ['transformer.h.0.mlp.c_proj.weight', 'transformer.h.0.mlp.c_proj.bias']
==================== Epoch: 0 ====================
Batch loss 8.195490837097168
Total loss 8.195490837097168
==================== Epoch: 1 ====================
Batch loss 8.172565460205078
Total loss 8.172565460205078
==================== Epoch: 2 ====================
Batch loss 3.324634075164795
Total loss 3.324634075164795
==================== Epoch: 3 ====================
Batch loss 2.1626291275024414
Total loss 2.1626291275024414
==================== Epoch: 4 ====================
Batch loss 1.5407875776290894
Total loss 1.5407875776290894
==================== Epoch: 5 ====================
Batch loss 1.3588957786560059
Total loss 1.3588957786560059
==================== Epoch: 6 ====================
Batch loss 1.0850945711135864
Total loss 1.0850945711135864
==================== Epo

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


Total loss 0.019478002563118935
==================== Epoch: 24 ====================
Batch loss 0.01717245765030384
Total loss 0.01717245765030384
Deltas successfully computed for ['transformer.h.0.mlp.c_proj.weight', 'transformer.h.0.mlp.c_proj.bias']
New weights successfully inserted into ['transformer.h.0.mlp.c_proj.weight', 'transformer.h.0.mlp.c_proj.bias']


A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


[cfact 20/20] reliability=0.5, locality=0.0
cfact phase done.


In [8]:
cfact_results

[{'edit_index': 1,
  'dataset_index': 12456,
  'edit_request': {'subject': 'Bani River',
                   'prompt': 'Bani River is located in',
                   'old_ans': 'Africa',
                   'new_ans': 'Antarctica'},
  'reliability': {'score': 0.5,
                  'details': [{'prompt': 'Bani River is located in',
                               'expected': 'Antarctica',
                               'generated': 'The Antarctic. It is the largest of the three',
                               'judge': 'yes'},
                              {'prompt': 'People around Bani River speak the language of',
                               'expected': 'Antarctica',
                               'generated': ' the people who live there. They are not the',
                               'judge': 'no'},
                              {'prompt': "Bani River's surroundings include",
                               'expected': 'Antarctica',
                               'generated': 'The

In [ ]:
filepath = save_results(
    {"cfact": cfact_results},
    editor.config.model_name,
    editor.config.alg_name,
    output_dir="../results/cfact",
)
print(f"Results saved to: {filepath}")

Results saved to: ../results/cfact\gpt2-xl_FT_2026-03-05_15-44.json


: 

In [ ]:
editor.close()
del editor